In [5]:
from pyflink.table import DataTypes, Schema, TableDescriptor
from pyflink.table import EnvironmentSettings, TableEnvironment

In [6]:
table_environment = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
configration = table_environment.get_config().get_configuration()
configration.set_string("execution.target", "remote")
configration.set_string("rest.address", "flink-jobmanager")
configration.set_integer("rest.port", 8082)
configration.set_string("execution.checkpointing.interval", "3 s")
configration.set_string("execution.checkpointing.mode", "EXACTLY_ONCE")
configration.set_string("execution.checkpointing.timeout", "10 min")
table_environment.get_config().get_configuration().set_string("parallelism.default", "1")

## Mysql -> Debezium -> Kafka -> Flink Source Table

In [7]:
table_environment.execute_sql("CREATE DATABASE IF NOT EXISTS mmix;")
table_environment.execute_sql("USE mmix;")
table_environment.execute_sql("DROP TABLE IF EXISTS event_log;")

In [8]:
source_event_log_descriptor = (
    TableDescriptor
    .for_connector("kafka")
    .schema(Schema.new_builder()
            .column("event_id", DataTypes.BIGINT())
            .column("event_uuid", DataTypes.STRING())
            .column("event_type", DataTypes.STRING())
            .column("event_time", DataTypes.BIGINT())
            .column("entity_id", DataTypes.BIGINT())
            .column("entity_type", DataTypes.STRING())
            .column("user_id", DataTypes.BIGINT().nullable())
            .column("payload", DataTypes.STRING().nullable())
            .build())
    .option("topic", "debezium.mysql.source.mmix.event_log")
    .option("properties.bootstrap.servers", "confluent-kafka-broker:19092")
    .option("properties.group.id", "mmix-debezium-mysql-source-flink")
    .option("scan.startup.mode", "earliest-offset")
    .option("format", "avro-confluent")
    .option("avro-confluent.schema-registry.url", "http://confluent-schema-registry:8081")
    .build())
table_environment.create_table("default_catalog.mmix.event_log", source_event_log_descriptor)
#table_environment.execute_sql("SELECT id, name, department, salary FROM debezium.mysql.source.mmix.source_event_log")

In [9]:
table_environment.execute_sql("CREATE CATALOG destination_catalog WITH ('type'='iceberg','catalog-type'='hadoop', 'warehouse'='s3a://mmix-prod-dataengineer-datalakehouse/streaming/destination')")
table_environment.execute_sql("USE CATALOG destination_catalog")
table_environment.execute_sql("CREATE DATABASE IF NOT EXISTS mmix")
table_environment.execute_sql("USE mmix")

2026-01-18 15:47:41,081 WARN  org.apache.hadoop.metrics2.impl.MetricsConfig                [] - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
2026-01-18 15:47:41,140 INFO  org.apache.hadoop.metrics2.impl.MetricsSystemImpl            [] - Scheduled Metric snapshot period at 10 second(s).
2026-01-18 15:47:41,141 INFO  org.apache.hadoop.metrics2.impl.MetricsSystemImpl            [] - s3a-file-system metrics system started


In [10]:
table_environment.execute_sql(
"""
CREATE TABLE IF NOT EXISTS destination_catalog.mmix.event_log
(
    event_id   BIGINT NOT NULL,
    event_uuid STRING,
    event_type STRING,
    event_time BIGINT,
    entity_id  BIGINT,
    entity_type STRING,
    user_id    BIGINT,
    payload STRING,
    PRIMARY KEY (event_id) NOT ENFORCED
)
WITH ('catalog-type'='hadoop', 'format-version'='2', 'write.upsert.enabled'='true')""")

In [12]:
table_environment.execute_sql("INSERT INTO destination_catalog.mmix.event_log SELECT event_id, event_uuid, event_type, event_time, entity_id, entity_type, user_id, payload FROM default_catalog.mmix.event_log;")